# Hardware-Aware Optimization: Both Rounds

This notebook compares the three hardware-aware optimizers across the original-circuit first round and the chained second round. Failure charts include every latest attempt. Performance charts use a matched set of circuits that completed all three first-round optimizers and all nine ordered two-step combinations, so each strategy is compared on the same circuits.

For second-round chains, reductions are measured from the original circuit and runtime includes both optimizer calls. The notebook also reports the second step's incremental gain and failures separately for each ordered pair. Original metrics are logical-circuit counts; mapped outputs can include routing overhead and physical-qubit expansion, so negative changes mean the mapped circuit has more gates or depth than that logical baseline.


In [ ]:
import sqlite3
from itertools import product
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda value: f'{value:,.2f}')

repo_root = Path.cwd()
if not (repo_root / 'dbs').exists():
    repo_root = repo_root.parent

FIRST_ROUND_DB = repo_root / 'dbs' / 'hardware_aware_combined' / 'first_round' / 'first_round.db'
SECOND_ROUND_DB = repo_root / 'dbs' / 'hardware_aware_combined' / 'second_round' / 'second_round.db'
for database_path in (FIRST_ROUND_DB, SECOND_ROUND_DB):
    if not database_path.exists():
        raise FileNotFoundError(f'Database not found: {database_path}')

OPTIMIZER_LABELS = {
    'qiskit_ai_hardware': 'Qiskit AI',
    'qiskit_standard_hardware': 'Qiskit standard',
    'tket_hardware': 'TKET',
}
OPTIMIZER_ORDER = list(OPTIMIZER_LABELS.values())
OPTIMIZER_COLORS = {
    'Qiskit AI': '#356F9F',
    'Qiskit standard': '#3C8D75',
    'TKET': '#C87943',
}
FAILURE_COLOR = '#B55252'
SUCCESS_COLOR = '#548B70'

sns.set_theme(
    style='whitegrid',
    context='notebook',
    rc={
        'figure.facecolor': '#FFFFFF',
        'axes.facecolor': '#FBFAF7',
        'axes.edgecolor': '#AAB3B9',
        'axes.labelcolor': '#26343D',
        'text.color': '#26343D',
        'xtick.color': '#34434C',
        'ytick.color': '#34434C',
        'grid.color': '#D8DEE2',
        'grid.linewidth': 0.7,
        'axes.titleweight': 'bold',
        'axes.titlepad': 12,
    },
)

print(f'First round:  {FIRST_ROUND_DB}')
print(f'Second round: {SECOND_ROUND_DB}')


In [ ]:
LATEST_RUNS_QUERY = '''
WITH ranked_runs AS (
    SELECT
        r.*,
        ROW_NUMBER() OVER (
            PARTITION BY r.circuit_id, r.optimizer_id
            ORDER BY r.id DESC
        ) AS run_rank
    FROM optimization_runs AS r
)
SELECT
    c.name AS circuit,
    c.category,
    c.source,
    c.num_qubits,
    o.name AS optimizer,
    r.success,
    r.error_message,
    r.duration_seconds,
    r.input_depth,
    r.output_depth,
    r.input_two_qubit_gates,
    r.output_two_qubit_gates
FROM ranked_runs AS r
JOIN circuits AS c ON c.id = r.circuit_id
JOIN optimizers AS o ON o.id = r.optimizer_id
WHERE r.run_rank = 1
'''


def load_latest_runs(database_path):
    with sqlite3.connect(database_path) as connection:
        frame = pd.read_sql_query(LATEST_RUNS_QUERY, connection)
    frame = frame[frame['optimizer'].isin(OPTIMIZER_LABELS)].copy()
    frame['optimizer_pretty'] = frame['optimizer'].map(OPTIMIZER_LABELS)
    frame['success'] = frame['success'].astype(bool)
    return frame


def add_reduction_metrics(frame, baseline_depth, baseline_twoq, final_depth, final_twoq):
    result = frame.copy()
    result['depth_reduction_pct'] = 100 * (
        result[baseline_depth] - result[final_depth]
    ) / result[baseline_depth].replace(0, np.nan)
    result['twoq_reduction_pct'] = 100 * (
        result[baseline_twoq] - result[final_twoq]
    ) / result[baseline_twoq].replace(0, np.nan)
    result.loc[~result['success'], ['depth_reduction_pct', 'twoq_reduction_pct']] = np.nan
    return result


round1_df = load_latest_runs(FIRST_ROUND_DB)
round1_df = add_reduction_metrics(
    round1_df,
    'input_depth', 'input_two_qubit_gates',
    'output_depth', 'output_two_qubit_gates',
)
round1_df['total_runtime_seconds'] = round1_df['duration_seconds']

round2_df = load_latest_runs(SECOND_ROUND_DB)
round2_df['original_circuit'] = round2_df['circuit'].str.rsplit('__', n=1).str[0]
round2_df['first_optimizer'] = round2_df['circuit'].str.rsplit('__', n=1).str[1]
round2_df['first_optimizer_pretty'] = round2_df['first_optimizer'].map(OPTIMIZER_LABELS)
round2_df['combination'] = (
    round2_df['first_optimizer_pretty'] + ' → ' + round2_df['optimizer_pretty']
)

parent_results = round1_df[[
    'circuit', 'optimizer', 'num_qubits', 'input_depth', 'input_two_qubit_gates',
    'duration_seconds', 'success',
]].rename(columns={
    'circuit': 'original_circuit',
    'optimizer': 'first_optimizer',
    'num_qubits': 'original_num_qubits',
    'input_depth': 'original_depth',
    'input_two_qubit_gates': 'original_two_qubit_gates',
    'duration_seconds': 'first_round_runtime_seconds',
    'success': 'first_round_success',
})
round2_df = round2_df.merge(
    parent_results,
    on=['original_circuit', 'first_optimizer'],
    how='left',
    validate='many_to_one',
)
if round2_df['first_round_success'].isna().any():
    raise ValueError('Could not match some second-round starting circuits to first-round results.')
if not round2_df['first_round_success'].all():
    raise ValueError('Second-round circuits include a first-round run that failed.')

round2_df['total_runtime_seconds'] = (
    round2_df['first_round_runtime_seconds'] + round2_df['duration_seconds']
)
round2_df['step_twoq_reduction_pct'] = 100 * (
    round2_df['input_two_qubit_gates'] - round2_df['output_two_qubit_gates']
) / round2_df['input_two_qubit_gates'].replace(0, np.nan)
round2_df.loc[
    round2_df['success'] & round2_df['input_two_qubit_gates'].eq(0),
    'step_twoq_reduction_pct',
] = 0.0
round2_df['step_depth_reduction_pct'] = 100 * (
    round2_df['input_depth'] - round2_df['output_depth']
) / round2_df['input_depth'].replace(0, np.nan)
round2_df = add_reduction_metrics(
    round2_df,
    'original_depth', 'original_two_qubit_gates',
    'output_depth', 'output_two_qubit_gates',
)
round2_df.loc[
    ~round2_df['success'], ['step_twoq_reduction_pct', 'step_depth_reduction_pct']
] = np.nan

COMBINATION_ORDER = [
    f'{first} → {second}'
    for first, second in product(OPTIMIZER_ORDER, repeat=2)
]

print(f'Round 1 latest attempts: {len(round1_df):,}')
print(f'Round 2 latest attempts: {len(round2_df):,}')


## Coverage and failures

These counts include every latest run. A second-round combination means the first optimizer that produced the starting circuit, followed by the optimizer applied in round two. Attempts differ by combination because round two only starts from successful first-round outputs.


In [ ]:
round1_outcomes = (
    round1_df.groupby('optimizer_pretty', as_index=False, observed=False)
    .agg(attempts=('success', 'size'), successes=('success', 'sum'))
)
round1_outcomes['failures'] = round1_outcomes['attempts'] - round1_outcomes['successes']
round1_outcomes['success_rate_pct'] = 100 * round1_outcomes['successes'] / round1_outcomes['attempts']
round1_outcomes['optimizer_pretty'] = pd.Categorical(
    round1_outcomes['optimizer_pretty'], OPTIMIZER_ORDER, ordered=True
)
round1_outcomes = round1_outcomes.sort_values('optimizer_pretty')

round2_outcomes = (
    round2_df.groupby('combination', as_index=False, observed=False)
    .agg(attempts=('success', 'size'), successes=('success', 'sum'))
)
round2_outcomes['failures'] = round2_outcomes['attempts'] - round2_outcomes['successes']
round2_outcomes['success_rate_pct'] = 100 * round2_outcomes['successes'] / round2_outcomes['attempts']
round2_outcomes['combination'] = pd.Categorical(
    round2_outcomes['combination'], COMBINATION_ORDER, ordered=True
)
round2_outcomes = round2_outcomes.sort_values('combination')

display(round1_outcomes)
display(round2_outcomes)

fig, axes = plt.subplots(1, 2, figsize=(17, 6), gridspec_kw={'width_ratios': [0.8, 1.8]})
for ax, summary, label_column, title in [
    (axes[0], round1_outcomes, 'optimizer_pretty', 'Round 1 · original → optimizer'),
    (axes[1], round2_outcomes, 'combination', 'Round 2 · first optimizer → second optimizer'),
]:
    labels = summary[label_column].astype(str).tolist()
    y = np.arange(len(summary))
    successes = summary['successes'].to_numpy()
    failures = summary['failures'].to_numpy()
    ax.barh(y, successes, color=SUCCESS_COLOR, label='Successful runs', height=0.72)
    ax.barh(y, failures, left=successes, color=FAILURE_COLOR, label='Failed runs', height=0.72)
    ax.set_yticks(y)
    ax.set_yticklabels(labels)
    ax.invert_yaxis()
    ax.set_xlabel('Latest attempts')
    ax.set_title(title, loc='left')
    ax.grid(axis='y', visible=False)
    ax.set_axisbelow(True)
    for index, row in enumerate(summary.itertuples(index=False)):
        ax.text(
            row.attempts + max(summary['attempts']) * 0.012,
            index,
            f'{row.failures} failed  ·  {row.success_rate_pct:.1f}% success',
            va='center', ha='left', fontsize=9, color='#34434C',
        )
    ax.set_xlim(0, max(summary['attempts']) * 1.42)
    for side in ('top', 'right', 'left'):
        ax.spines[side].set_visible(False)
    ax.tick_params(axis='y', length=0)
axes[1].legend(frameon=False, loc='lower right')
fig.suptitle('Run completion by optimizer choice', x=0.03, ha='left', fontsize=16, fontweight='bold')
fig.tight_layout()
plt.show()
plt.close(fig)



## First round performance

The matched set includes circuits for which all three first-round optimizers succeeded. Each optimizer is measured against the same original circuits. Reductions above zero indicate fewer gates or lower depth than the original; negative values indicate a regression.


In [ ]:
round1_circuit_status = round1_df.groupby('circuit').agg(
    optimizer_count=('optimizer', 'nunique'), all_succeeded=('success', 'all')
)
matched_round1_circuits = round1_circuit_status.index[
    (round1_circuit_status['optimizer_count'] == len(OPTIMIZER_ORDER))
    & round1_circuit_status['all_succeeded']
]
round1_matched = round1_df[round1_df['circuit'].isin(matched_round1_circuits)].copy()
print(f'Matched first-round circuits: {len(matched_round1_circuits):,}')

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
plot_specs = [
    ('twoq_reduction_pct', '2Q gate reduction from original (%)', False),
    ('depth_reduction_pct', 'Depth reduction from original (%)', False),
    ('total_runtime_seconds', 'Optimizer runtime (seconds)', True),
]
for ax, (column, xlabel, log_scale) in zip(axes, plot_specs):
    sns.boxplot(
        data=round1_matched,
        x=column,
        y='optimizer_pretty',
        order=OPTIMIZER_ORDER,
        hue='optimizer_pretty',
        hue_order=OPTIMIZER_ORDER,
        palette=OPTIMIZER_COLORS,
        showfliers=False,
        width=0.62,
        ax=ax,
    )
    if ax.get_legend() is not None:
        ax.get_legend().remove()
    ax.axvline(0, color='#59666E', linewidth=1, linestyle='--', alpha=0.7)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('')
    ax.set_title(xlabel.split(' from ')[0] if ' from ' in xlabel else 'Runtime distribution', loc='left')
    if log_scale:
        ax.set_xscale('log')
    ax.grid(axis='y', visible=False)
    for side in ('top', 'right', 'left'):
        ax.spines[side].set_visible(False)
    ax.tick_params(axis='y', length=0)
fig.suptitle('Round 1 · per-circuit results on the matched set', x=0.03, ha='left', fontsize=16, fontweight='bold')
fig.tight_layout()
plt.show()
plt.close(fig)


In [ ]:
def best_result_counts(frame, category_column, category_order, metrics):
    records = []
    for metric_label, value_column in metrics.items():
        values = frame.dropna(subset=[value_column]).copy()
        best_values = values.groupby('circuit')[value_column].transform('min')
        winners = values.loc[values[value_column].eq(best_values)]
        counts = winners.groupby(category_column).size().reindex(category_order, fill_value=0)
        for category, count in counts.items():
            records.append({'metric': metric_label, 'strategy': category, 'wins': int(count)})
    return pd.DataFrame(records)


ROUND1_WIN_METRICS = {
    '2Q gates': 'output_two_qubit_gates',
    'Depth': 'output_depth',
    'Runtime': 'total_runtime_seconds',
}
round1_wins = best_result_counts(
    round1_matched, 'optimizer_pretty', OPTIMIZER_ORDER, ROUND1_WIN_METRICS
)
display(round1_wins.pivot(index='strategy', columns='metric', values='wins'))

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
for ax, metric in zip(axes, ROUND1_WIN_METRICS):
    subset = round1_wins[round1_wins['metric'] == metric]
    colors = [OPTIMIZER_COLORS[name] for name in subset['strategy']]
    ax.barh(subset['strategy'], subset['wins'], color=colors, height=0.68)
    ax.set_title(f'{metric} · lower is better', loc='left')
    ax.set_xlabel('Circuit wins (ties count for each winner)')
    ax.grid(axis='y', visible=False)
    for side in ('top', 'right', 'left'):
        ax.spines[side].set_visible(False)
    ax.tick_params(axis='y', length=0)
axes[0].invert_yaxis()
fig.suptitle('Round 1 · best-result counts on matched circuits', x=0.03, ha='left', fontsize=16, fontweight='bold')
fig.tight_layout()
plt.show()
plt.close(fig)


## Second round combination matrix

Each cell is an ordered pair: the row optimizer created the starting circuit, then the column optimizer ran in round two. The quality heatmaps use only the fully matched circuit set and report the final result versus the original. Failure counts and success rates use all available attempts.


In [ ]:
round2_circuit_status = round2_df.groupby('original_circuit').agg(
    combination_count=('combination', 'nunique'), all_succeeded=('success', 'all')
)
matched_round2_circuits = round2_circuit_status.index[
    (round2_circuit_status['combination_count'] == len(COMBINATION_ORDER))
    & round2_circuit_status['all_succeeded']
]
round2_matched = round2_df[
    round2_df['original_circuit'].isin(matched_round2_circuits)
].copy()
print(f'Matched complete chains: {len(matched_round2_circuits):,} original circuits')

round2_matrix_rows = OPTIMIZER_ORDER
round2_matrix_columns = OPTIMIZER_ORDER
failure_grid = round2_df.pivot_table(
    index='first_optimizer_pretty', columns='optimizer_pretty',
    values='success', aggfunc=lambda values: int((~values).sum()),
).reindex(index=round2_matrix_rows, columns=round2_matrix_columns)
attempt_grid = round2_df.pivot_table(
    index='first_optimizer_pretty', columns='optimizer_pretty',
    values='success', aggfunc='size',
).reindex(index=round2_matrix_rows, columns=round2_matrix_columns)
success_rate_grid = round2_df.pivot_table(
    index='first_optimizer_pretty', columns='optimizer_pretty',
    values='success', aggfunc='mean',
).reindex(index=round2_matrix_rows, columns=round2_matrix_columns) * 100

quality_metrics = [
    ('twoq_reduction_pct', 'Mean 2Q reduction from original (%)', '.1f'),
    ('depth_reduction_pct', 'Mean depth reduction from original (%)', '.1f'),
    ('total_runtime_seconds', 'Median total runtime, both rounds (s)', '.1f'),
]
quality_grids = {
    column: round2_matched.pivot_table(
        index='first_optimizer_pretty', columns='optimizer_pretty', values=column, aggfunc='mean'
    ).reindex(index=round2_matrix_rows, columns=round2_matrix_columns)
    for column, _, _ in quality_metrics
}
quality_grids['total_runtime_seconds'] = round2_matched.pivot_table(
    index='first_optimizer_pretty', columns='optimizer_pretty',
    values='total_runtime_seconds', aggfunc='median',
).reindex(index=round2_matrix_rows, columns=round2_matrix_columns)

fig, axes = plt.subplots(2, 3, figsize=(17, 10))
sns.heatmap(
    failure_grid, annot=True, fmt='.0f', cmap='Reds', linewidths=1,
    linecolor='#FFFFFF', cbar_kws={'label': 'Failed runs'}, ax=axes[0, 0],
)
axes[0, 0].set_title('Failures · all attempts', loc='left')
sns.heatmap(
    success_rate_grid, annot=True, fmt='.1f', cmap='YlGn', vmin=90, vmax=100,
    linewidths=1, linecolor='#FFFFFF', cbar_kws={'label': 'Success rate (%)'}, ax=axes[0, 1],
)
axes[0, 1].set_title('Success rate · all attempts', loc='left')
sns.heatmap(
    attempt_grid, annot=True, fmt='.0f', cmap='Blues', linewidths=1,
    linecolor='#FFFFFF', cbar_kws={'label': 'Latest attempts'}, ax=axes[0, 2],
)
axes[0, 2].set_title('Attempts · all runs', loc='left')
for ax, (column, title, fmt) in zip(axes[1], quality_metrics):
    grid = quality_grids[column]
    sns.heatmap(
        grid, annot=True, fmt=fmt, cmap='crest', linewidths=1,
        linecolor='#FFFFFF', cbar_kws={'label': title}, ax=ax,
    )
    ax.set_title(title, loc='left')
for ax in axes.flat:
    if ax.has_data():
        ax.set_xlabel('Round 2 optimizer')
        ax.set_ylabel('Round 1 optimizer')
fig.suptitle('Round 2 · combination-by-combination results', x=0.03, ha='left', fontsize=16, fontweight='bold')
fig.tight_layout()
plt.show()
plt.close(fig)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 9), sharey=True)
round2_plot_specs = [
    ('twoq_reduction_pct', 'Total 2Q reduction from original (%)', False),
    ('depth_reduction_pct', 'Total depth reduction from original (%)', False),
    ('total_runtime_seconds', 'Total runtime, both rounds (seconds)', True),
]
for ax, (column, xlabel, log_scale) in zip(axes, round2_plot_specs):
    sns.boxplot(
        data=round2_matched,
        x=column,
        y='combination',
        order=COMBINATION_ORDER,
        color='#7D91A0',
        showfliers=False,
        width=0.68,
        ax=ax,
    )
    if not log_scale:
        ax.axvline(0, color='#59666E', linewidth=1, linestyle='--', alpha=0.7)
    else:
        ax.set_xscale('log')
    ax.set_xlabel(xlabel)
    ax.set_ylabel('' if ax is not axes[0] else 'Round 1 → Round 2')
    ax.set_title(xlabel.split(' from ')[0] if ' from ' in xlabel else 'Combined runtime', loc='left')
    ax.grid(axis='y', visible=False)
    for side in ('top', 'right', 'left'):
        ax.spines[side].set_visible(False)
    ax.tick_params(axis='y', length=0, labelleft=ax is axes[0])
fig.suptitle('Round 2 · per-circuit chain outcomes on the matched set', x=0.03, ha='left', fontsize=16, fontweight='bold')
fig.tight_layout()
plt.show()
plt.close(fig)


In [ ]:
ROUND2_WIN_METRICS = {
    '2Q gates': 'output_two_qubit_gates',
    'Depth': 'output_depth',
    'Total runtime': 'total_runtime_seconds',
}
round2_wins = best_result_counts(
    round2_matched,
    'combination',
    COMBINATION_ORDER,
    ROUND2_WIN_METRICS,
)
display(round2_wins.pivot(index='strategy', columns='metric', values='wins'))

fig, axes = plt.subplots(1, 3, figsize=(19, 8), sharey=True)
first_optimizer_for_combo = {
    combination: combination.split(' → ')[0] for combination in COMBINATION_ORDER
}
for ax, metric in zip(axes, ROUND2_WIN_METRICS):
    subset = round2_wins[round2_wins['metric'] == metric]
    colors = [OPTIMIZER_COLORS[first_optimizer_for_combo[name]] for name in subset['strategy']]
    ax.barh(subset['strategy'], subset['wins'], color=colors, height=0.7)
    ax.set_title(f'{metric} · lower is better', loc='left')
    ax.set_xlabel('Circuit wins (ties count for each winner)')
    ax.grid(axis='y', visible=False)
    for side in ('top', 'right', 'left'):
        ax.spines[side].set_visible(False)
    ax.tick_params(axis='y', length=0, labelleft=ax is axes[0])
axes[0].invert_yaxis()
fig.suptitle('Round 2 · best chain counts on matched circuits', x=0.03, ha='left', fontsize=16, fontweight='bold')
fig.tight_layout()
plt.show()
plt.close(fig)


## Which single optimizer or chain wins most often?

This compares all three single optimizers and all nine ordered two-optimizer chains on one shared complete-case set. A win means the strategy produced the fewest final two-qubit gates for that circuit; ties count for every tied strategy.

In [ ]:
# Compare all 3 single optimizers and all 9 ordered chains on the same circuits.
joint_circuits = sorted(
    set(round1_matched['circuit']) & set(round2_matched['original_circuit'])
)
round1_joint = round1_matched[round1_matched['circuit'].isin(joint_circuits)]
round2_joint = round2_matched[round2_matched['original_circuit'].isin(joint_circuits)]

single_results = round1_joint[[
    'circuit', 'optimizer_pretty', 'output_two_qubit_gates'
]].rename(columns={'optimizer_pretty': 'strategy'})
chain_results = round2_joint[[
    'original_circuit', 'combination', 'output_two_qubit_gates'
]].rename(columns={
    'original_circuit': 'circuit',
    'combination': 'strategy',
})
candidate_results = pd.concat([single_results, chain_results], ignore_index=True)
JOINT_STRATEGY_ORDER = OPTIMIZER_ORDER + COMBINATION_ORDER

best_gate_count = candidate_results.groupby('circuit')['output_two_qubit_gates'].transform('min')
winner_rows = candidate_results.loc[
    candidate_results['output_two_qubit_gates'].eq(best_gate_count)
]
joint_win_counts = (
    winner_rows.groupby('strategy').size()
    .reindex(JOINT_STRATEGY_ORDER, fill_value=0)
    .astype(int)
)

print(f'Circuits with all 12 successful strategies: {len(joint_circuits):,}')
print(f'Winner counts include ties: {len(winner_rows):,} strategy-circuit wins')

outcome_wide = candidate_results.pivot(
    index='circuit', columns='strategy', values='output_two_qubit_gates'
)
best_single_gates = outcome_wide[OPTIMIZER_ORDER].min(axis=1)
best_chain_gates = outcome_wide[COMBINATION_ORDER].min(axis=1)
original_gates = round1_joint.groupby('circuit')['input_two_qubit_gates'].first()
print(
    f'A chain strictly beats the best single optimizer on '
    f'{(best_chain_gates < best_single_gates).sum():,}/{len(joint_circuits):,} circuits.'
)
print(
    f'At least one chain reduces two-qubit gates versus the original on '
    f'{(best_chain_gates < original_gates).sum():,}/{len(joint_circuits):,} circuits.'
)

# Lollipop marks keep the 12-way comparison compact and distinct from the
# adjacent heatmaps and distribution plots. Marker shape distinguishes singles
# from two-step chains; every row uses the same complete-case circuit set.
strategies = joint_win_counts.index.tolist()
win_values = joint_win_counts.to_numpy()
positions = np.arange(len(strategies))
is_chain = np.array([' → ' in strategy for strategy in strategies])
fig, ax = plt.subplots(figsize=(11, 7.5))
ax.hlines(positions, 0, win_values, color='#CDD5D9', linewidth=1.4, zorder=1)
for mask, label, color, marker in [
    (~is_chain, 'Single optimizer', '#3C8D75', 'o'),
    (is_chain, 'Two-optimizer chain', '#356F9F', 'D'),
]:
    ax.scatter(
        win_values[mask], positions[mask],
        color=color, marker=marker, s=58, label=label, zorder=2,
    )
for position, value in zip(positions, win_values):
    ax.annotate(
        f'{value}', (value, position), xytext=(6, 0),
        textcoords='offset points', va='center', ha='left', fontsize=9,
    )
ax.set_yticks(positions)
ax.set_yticklabels(strategies)
ax.invert_yaxis()
ax.set_xlim(0, max(1, win_values.max()) * 1.16)
ax.set_xlabel('Circuits tied for fewest final two-qubit gates')
ax.set_ylabel('Strategy')
ax.set_title('Single optimizers and ordered chains · matched complete cases', loc='left')
ax.grid(axis='y', visible=False)
ax.set_axisbelow(True)
for side in ('top', 'right', 'left'):
    ax.spines[side].set_visible(False)
ax.tick_params(axis='y', length=0)
ax.legend(frameon=False, loc='lower right')
fig.tight_layout()
plt.show()
plt.close(fig)

## Step-two incremental gains

This last comparison isolates what the second optimizer changed relative to its input circuit. Positive values mean the second step reduced the metric; negative values mean it increased the metric. This helps distinguish chains that are strong overall from a second optimizer that adds little or reverses the first optimizer's gains.


In [ ]:
incremental_summary = (
    round2_matched.groupby('combination', observed=False, as_index=False)
    .agg(
        median_step_twoq_reduction_pct=('step_twoq_reduction_pct', 'median'),
        median_step_depth_reduction_pct=('step_depth_reduction_pct', 'median'),
        median_round2_runtime_seconds=('duration_seconds', 'median'),
    )
)
incremental_summary['combination'] = pd.Categorical(
    incremental_summary['combination'], COMBINATION_ORDER, ordered=True
)
incremental_summary = incremental_summary.sort_values('combination')
display(incremental_summary)

fig, axes = plt.subplots(1, 2, figsize=(16, 8), sharey=True)
for ax, column, title in [
    (axes[0], 'median_step_twoq_reduction_pct', 'Median 2Q reduction in round 2 (%)'),
    (axes[1], 'median_step_depth_reduction_pct', 'Median depth reduction in round 2 (%)'),
]:
    values = incremental_summary[column]
    colors = [OPTIMIZER_COLORS[name.split(' → ')[0]] for name in incremental_summary['combination'].astype(str)]
    ax.barh(incremental_summary['combination'].astype(str), values, color=colors, height=0.7)
    for row_index, value in enumerate(values):
        if pd.notna(value):
            ax.annotate(
                f'{value:.1f}', (value, row_index), xytext=(4 if value >= 0 else -4, 0),
                textcoords='offset points', va='center',
                ha='left' if value >= 0 else 'right', fontsize=9, color='#26343D',
            )
    ax.axvline(0, color='#59666E', linewidth=1, linestyle='--', alpha=0.7)
    ax.set_title(title, loc='left')
    ax.set_xlabel('Incremental reduction vs. round 2 input (%)')
    ax.grid(axis='y', visible=False)
    for side in ('top', 'right', 'left'):
        ax.spines[side].set_visible(False)
    ax.tick_params(axis='y', length=0, labelleft=ax is axes[0])
axes[0].invert_yaxis()
fig.suptitle('Round 2 · marginal benefit from the second optimizer', x=0.03, ha='left', fontsize=16, fontweight='bold')
fig.tight_layout()
plt.show()
plt.close(fig)


## Paper figure: two-qubit gate and depth reduction

Two aligned horizontal bar plots compare mean two-qubit gate reduction and mean depth reduction for the same 646 circuits. Bars show the mean percentage reduction from the original circuit; numerical labels give the mean values. Colors identify the first optimizer, and hatching denotes ordered two-step chains. Negative reductions indicate increases in the metric.


In [ ]:
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# Load the complete master directly so this figure is reproducible independently.
figure_repo_root = Path.cwd()
while not (figure_repo_root / 'dbs' / 'hardware_aware_combined' / 'combined_master.csv').exists():
    if figure_repo_root.parent == figure_repo_root:
        raise FileNotFoundError('Could not locate the hardware-aware combined master.')
    figure_repo_root = figure_repo_root.parent
master = pd.read_csv(figure_repo_root / 'dbs' / 'hardware_aware_combined' / 'combined_master.csv')
master = master.loc[~master['error'].astype(bool)].copy()

optimizer_names_raw = ['qiskit_ai', 'qiskit_standard', 'tket']
optimizer_labels = {
    'qiskit_ai': 'Qiskit AI',
    'qiskit_standard': 'Qiskit standard',
    'tket': 'TKET',
}
combination_names_raw = [f'{first}__{second}' for first in optimizer_names_raw for second in optimizer_names_raw]
strategy_names_raw = optimizer_names_raw + combination_names_raw
master = master.loc[master['chain_name'].isin(strategy_names_raw)].copy()

# Require all 12 successful outcomes per circuit and a nonzero logical 2Q baseline.
circuit_coverage = master.groupby('original_circuit').agg(
    strategy_count=('chain_name', 'nunique'),
    initial_two_qubit_gates=('initial_two_qubit_gates', 'first'),
)
matched_circuits = circuit_coverage.index[
    circuit_coverage['strategy_count'].eq(12)
    & circuit_coverage['initial_two_qubit_gates'].gt(0)
].sort_values()
master = master.loc[master['original_circuit'].isin(matched_circuits)].copy()
master['strategy'] = master['chain_name'].map(
    lambda value: ' → '.join(optimizer_labels[part] for part in value.split('__'))
)
master['two_qubit_reduction_pct'] = 100 * (
    master['initial_two_qubit_gates'] - master['final_two_qubit_gates']
) / master['initial_two_qubit_gates']
master['depth_reduction_pct'] = 100 * (
    master['initial_depth'] - master['final_depth']
) / master['initial_depth'].replace(0, np.nan)

strategy_order_raw = optimizer_names_raw + combination_names_raw
strategy_order = [
    ' → '.join(optimizer_labels[part] for part in name.split('__'))
    for name in strategy_order_raw
]
strategy_outcomes = master.copy()
if len(matched_circuits) != 646 or strategy_outcomes.groupby('strategy')['original_circuit'].nunique().ne(646).any():
    raise ValueError(f'Expected 646 matched circuits for each strategy; found {len(matched_circuits)}.')

# Paired bootstrap: each draw resamples the same circuits for all methods and both metrics.
rng = np.random.default_rng(20261010)
metric_columns = ['two_qubit_reduction_pct', 'depth_reduction_pct']
wide_metrics = {
    metric: strategy_outcomes.pivot(index='original_circuit', columns='strategy', values=metric)
        .reindex(index=matched_circuits, columns=strategy_order)
    for metric in metric_columns
}
metric_values = {metric: wide.to_numpy(dtype=float) for metric, wide in wide_metrics.items()}
if any(not np.isfinite(values).all() for values in metric_values.values()):
    raise ValueError('Missing gate or depth values in the matched strategy set.')
sample_indices = rng.integers(0, len(matched_circuits), size=(10000, len(matched_circuits)))
summary_rows = []
for metric in metric_columns:
    values = metric_values[metric]
    boot_means = values[sample_indices].mean(axis=1)
    means = values.mean(axis=0)
    ci_low, ci_high = np.percentile(boot_means, [2.5, 97.5], axis=0)
    for strategy, mean, low, high in zip(strategy_order, means, ci_low, ci_high):
        summary_rows.append({
            'strategy': strategy,
            'metric': metric,
            'mean_reduction_pct': float(mean),
            'ci_95_low': float(low),
            'ci_95_high': float(high),
            'circuits': len(matched_circuits),
        })
strategy_summary_646 = pd.DataFrame(summary_rows)

color_by_optimizer = {
    'Qiskit AI': '#356F9F',
    'Qiskit standard': '#3C8D75',
    'TKET': '#C87943',
}
first_optimizer = {
    name: (name.split(' → ')[0] if ' → ' in name else name)
    for name in strategy_order
}
strategy_kind = {
    name: ('Two-step chain' if ' → ' in name else 'Single optimizer')
    for name in strategy_order
}
fig, axes = plt.subplots(1, 2, figsize=(6.9, 8.6), sharey=True, facecolor='#FFFFFF')
plot_specs = [
    ('two_qubit_reduction_pct', '2Q gate\nreduction (%)'),
    ('depth_reduction_pct', 'Depth\nreduction (%)'),
]
y = np.arange(len(strategy_order))
for ax, (metric, title) in zip(axes, plot_specs):
    data = strategy_summary_646.loc[strategy_summary_646['metric'].eq(metric)].set_index('strategy').loc[strategy_order]
    for row_i, strategy in enumerate(strategy_order):
        row = data.loc[strategy]
        color = color_by_optimizer[first_optimizer[strategy]]
        is_chain = strategy_kind[strategy] == 'Two-step chain'
        ax.barh(
            row_i, row['mean_reduction_pct'], left=0, height=0.56,
            color=color, edgecolor=color, linewidth=0.45, alpha=0.72,
            hatch='///' if is_chain else None, zorder=2,
        )
        value = row['mean_reduction_pct']
        value_label = f"{value:.1f}%"
        metric_span = data['mean_reduction_pct'].abs().max()
        if abs(value) < 0.30 * metric_span:
            # Keep labels readable when a bar is too short to contain the text.
            ax.text(
                value - 0.018 * metric_span, row_i, value_label,
                ha='right', va='center', fontsize=13.0, color='black', zorder=5,
            )
        else:
            ax.text(
                value * 0.5, row_i, value_label,
                ha='center', va='center', fontsize=13.0, color='black', zorder=5,
            )
    ax.axvline(0, color='#657681', linewidth=0.8, linestyle='--', zorder=1)
    ax.set_title(title, fontsize=15.5, weight='bold', color='#24333D', pad=8)
    ax.set_xlabel('Reduction (%)', fontsize=13.5, color='#24333D')
    ax.set_axisbelow(True)
    ax.set_facecolor('#F7F4EA')
    ax.grid(axis='x', color='#CDD5D9', linewidth=0.55, alpha=0.9)
    ax.grid(axis='y', visible=False)
    for spine in ax.spines.values():
        spine.set_color('#AAB3B9')
        spine.set_linewidth(0.6)
    ax.tick_params(axis='x', labelsize=12.5, colors='#24333D')
    ax.tick_params(axis='y', length=0, labelsize=20, colors='#24333D')
    ax.set_ylim(len(strategy_order) - 0.35, -0.65)
axes[0].set_yticks(y)
axes[0].set_yticklabels(strategy_order, fontsize=20.0)
axes[0].set_ylabel('')
color_handles = [
    Patch(facecolor=color, edgecolor=color, alpha=0.72, label=name)
    for name, color in color_by_optimizer.items()
]
kind_handles = [
    Patch(facecolor='#AAB3B9', edgecolor='#657681', label='Single optimizer'),
    Patch(facecolor='#AAB3B9', edgecolor='#657681', hatch='///', label='Two-step chain'),
]
fig.legend(handles=color_handles + kind_handles, loc='lower center', ncol=3,
           frameon=False, fontsize=15.0, bbox_to_anchor=(0.62, 0.0),
           handletextpad=0.35, columnspacing=0.85)
fig.suptitle('Hardware-aware Optimizer Results',
             x=0.61, y=0.985, ha='center', fontsize=26.0, weight='bold', color='#24333D')
fig.subplots_adjust(left=0.41, right=0.99, top=0.88, bottom=0.14, wspace=0.28)

figure_dir = figure_repo_root / 'data' / 'analyze_hardware_data_figures'
figure_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(figure_dir / 'hardware_strategy_performance_646.pdf', bbox_inches='tight', dpi=300, facecolor=fig.get_facecolor())
fig.savefig(figure_dir / 'hardware_strategy_performance_646.png', bbox_inches='tight', dpi=300, facecolor=fig.get_facecolor())
strategy_summary_646.to_csv(figure_dir / 'hardware_strategy_performance_646_summary.csv', index=False)
display(strategy_summary_646.pivot(index='strategy', columns='metric', values='mean_reduction_pct').reindex(strategy_order))
plt.show()
plt.close(fig)


## Best-result counts by method

For each circuit, this counts every method tied for the largest reduction from the original circuit. Because every method for a circuit uses the same original baseline, the largest reduction is equivalent to the smallest final gate count or depth. Bars can sum to more than the circuit count because ties count for every tied method.

In [ ]:
# Count per-circuit best results from the same 646-circuit master used by the
# hardware-aware summary figure above. Ties are credited to every tied method.
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

figure_repo_root = Path.cwd()
while not (figure_repo_root / "dbs" / "hardware_aware_combined" / "combined_master.csv").exists():
    if figure_repo_root.parent == figure_repo_root:
        raise FileNotFoundError("Could not locate the hardware-aware combined master.")
    figure_repo_root = figure_repo_root.parent

wins_master = pd.read_csv(
    figure_repo_root / "dbs" / "hardware_aware_combined" / "combined_master.csv"
)
wins_master = wins_master.loc[~wins_master["error"].astype(bool)].copy()
wins_master["two_qubit_reduction_pct"] = 100 * (
    wins_master["initial_two_qubit_gates"] - wins_master["final_two_qubit_gates"]
) / wins_master["initial_two_qubit_gates"]
wins_master["depth_reduction_pct"] = 100 * (
    wins_master["initial_depth"] - wins_master["final_depth"]
) / wins_master["initial_depth"].replace(0, np.nan)

optimizer_names_raw = ["qiskit_ai", "qiskit_standard", "tket"]
optimizer_labels = {
    "qiskit_ai": "Qiskit AI",
    "qiskit_standard": "Qiskit standard",
    "tket": "TKET",
}
combination_names_raw = [
    f"{first}__{second}"
    for first in optimizer_names_raw
    for second in optimizer_names_raw
]
strategy_names_raw = optimizer_names_raw + combination_names_raw
strategy_order = [
    " → ".join(optimizer_labels[part] for part in strategy.split("__"))
    for strategy in strategy_names_raw
]

if wins_master.groupby("original_circuit")["chain_name"].nunique().ne(12).any():
    raise ValueError("Expected all 12 methods for every circuit in the complete-case master.")

win_rows = []
for metric, column in [
    ("2Q gate reduction", "two_qubit_reduction_pct"),
    ("Depth reduction", "depth_reduction_pct"),
]:
    values = wins_master.pivot(
        index="original_circuit", columns="chain_name", values=column
    ).reindex(columns=strategy_names_raw)
    if values.isna().any().any():
        raise ValueError(f"Missing {metric} values in the matched method set.")
    best = values.max(axis=1)
    tied_for_best = np.isclose(
        values.to_numpy(dtype=float),
        best.to_numpy(dtype=float)[:, None],
        rtol=0,
        atol=1e-10,
    )
    counts = pd.Series(tied_for_best.sum(axis=0), index=strategy_names_raw)
    win_rows.extend(
        {"strategy": label, "metric": metric, "wins": int(counts[raw])}
        for raw, label in zip(strategy_names_raw, strategy_order)
    )

method_wins = pd.DataFrame(win_rows)
positions = np.arange(len(strategy_order))
bar_height = 0.34
fig, ax = plt.subplots(figsize=(9.5, 10), facecolor="#FFFFFF")
ax.set_facecolor("#F7F4EA")
for offset, metric, color in [
    (-bar_height / 2, "2Q gate reduction", "#356F9F"),
    (bar_height / 2, "Depth reduction", "#C87943"),
]:
    counts = (
        method_wins.loc[method_wins["metric"].eq(metric)]
        .set_index("strategy")["wins"]
        .reindex(strategy_order)
    )
    bars = ax.barh(
        positions + offset,
        counts.to_numpy(),
        height=bar_height,
        color=color,
        edgecolor=color,
        linewidth=0.45,
        alpha=0.78,
        label=metric,
        zorder=2,
    )
    ax.bar_label(bars, padding=3, fontsize=15, color="#24333D")

ax.set_yticks(positions)
ax.set_yticklabels(strategy_order)
ax.invert_yaxis()
ax.set_xlabel("Circuits where method was best", fontsize=20, color="#24333D")
ax.set_title(
    "Best Result Counts by Method",
    loc="center",
    fontsize=35,
    weight="bold",
    color="#24333D",
    pad=10,
)
ax.set_axisbelow(True)
ax.grid(axis="x", color="#CDD5D9", linewidth=0.55, alpha=0.9)
ax.grid(axis="y", visible=False)
for spine in ax.spines.values():
    spine.set_color("#AAB3B9")
    spine.set_linewidth(0.6)
ax.tick_params(axis="x", labelsize=15, colors="#24333D")
ax.tick_params(axis="y", length=0, labelsize=25, colors="#24333D")
ax.set_xlim(0, max(method_wins["wins"]) * 1.16)
fig.legend(
    loc="lower center",
    ncol=2,
    frameon=False,
    fontsize=15,
    bbox_to_anchor=(0.66, -0.02),
    handletextpad=0.45,
    columnspacing=1.2,
)
fig.subplots_adjust(left=0.36, right=0.98, top=0.92, bottom=0.08)
figure_dir = Path("analyze_hardware_data_figures")
figure_dir.mkdir(parents=True, exist_ok=True)

fig.savefig(
    figure_dir / "hardware_aware_method_best_counts.png",
    dpi=300,
    bbox_inches="tight",
    facecolor=fig.get_facecolor(),
)
plt.show()
plt.close(fig)

## Fixed and oracle optimizer comparison

This table compares the best fixed single optimizer and fixed two-optimizer chain with per-circuit oracle choices. Each value is the mean per-circuit percentage reduction across circuits with all 12 successful methods and a nonzero initial two-qubit-gate count. The oracle selects the best single optimizer or chain separately for each circuit and metric. Negative values mean the final circuit metric was larger than its baseline on average.

In [ ]:
# Compare fixed strategies with per-circuit oracle choices on the same complete-case set.
from pathlib import Path

import pandas as pd
from IPython.display import display

comparison_repo_root = Path.cwd()
while not (comparison_repo_root / "dbs" / "hardware_aware_combined" / "combined_master.csv").exists():
    if comparison_repo_root.parent == comparison_repo_root:
        raise FileNotFoundError("Could not locate the hardware-aware combined master.")
    comparison_repo_root = comparison_repo_root.parent

comparison_master = pd.read_csv(
    comparison_repo_root / "dbs" / "hardware_aware_combined" / "combined_master.csv"
)
comparison_master = comparison_master.loc[~comparison_master["error"].astype(bool)].copy()

individual_methods = ["qiskit_ai", "qiskit_standard", "tket"]
all_methods = individual_methods + [
    f"{first}__{second}"
    for first in individual_methods
    for second in individual_methods
]
comparison_master = comparison_master.loc[
    comparison_master["chain_name"].isin(all_methods)
].copy()

# Match the 646-circuit population used by the hardware-aware best-count chart:
# all 12 methods must be present, and the 2Q baseline must be nonzero.
circuit_coverage = comparison_master.groupby("original_circuit").agg(
    method_count=("chain_name", "nunique"),
    initial_two_qubit_gates=("initial_two_qubit_gates", "first"),
    initial_depth=("initial_depth", "first"),
)
matched_circuits = circuit_coverage.index[
    circuit_coverage["method_count"].eq(len(all_methods))
    & circuit_coverage["initial_two_qubit_gates"].gt(0)
    & circuit_coverage["initial_depth"].gt(0)
]
comparison_master = comparison_master.loc[
    comparison_master["original_circuit"].isin(matched_circuits)
].copy()

if comparison_master.groupby("original_circuit")["chain_name"].nunique().ne(12).any():
    raise ValueError("The comparison set does not contain all 12 methods per circuit.")

comparison_master["two_qubit_reduction_pct"] = 100 * (
    comparison_master["initial_two_qubit_gates"]
    - comparison_master["final_two_qubit_gates"]
) / comparison_master["initial_two_qubit_gates"]
comparison_master["depth_reduction_pct"] = 100 * (
    comparison_master["initial_depth"] - comparison_master["final_depth"]
) / comparison_master["initial_depth"]

individual_frame = comparison_master.loc[
    comparison_master["chain_name"].isin(individual_methods)
]
chain_frame = comparison_master.loc[
    ~comparison_master["chain_name"].isin(individual_methods)
]

def best_fixed_mean(frame, metric):
    # Average each fixed method across the same circuits, then take the best mean.
    per_method_means = frame.groupby("chain_name")[metric].mean()
    return per_method_means.max()

def oracle_mean(frame, metric):
    # For each circuit, take the best method in this class, then average those results.
    return frame.groupby("original_circuit")[metric].max().mean()

comparison_table = pd.DataFrame(
    {
        "Method class": [
            "Best fixed individual optimizer",
            "Best fixed two-optimizer chain",
            "Oracle individual optimizer",
            "Oracle two-optimizer chain",
        ],
        "2Q reduction (%)": [
            best_fixed_mean(individual_frame, "two_qubit_reduction_pct"),
            best_fixed_mean(chain_frame, "two_qubit_reduction_pct"),
            oracle_mean(individual_frame, "two_qubit_reduction_pct"),
            oracle_mean(chain_frame, "two_qubit_reduction_pct"),
        ],
        "Depth reduction (%)": [
            best_fixed_mean(individual_frame, "depth_reduction_pct"),
            best_fixed_mean(chain_frame, "depth_reduction_pct"),
            oracle_mean(individual_frame, "depth_reduction_pct"),
            oracle_mean(chain_frame, "depth_reduction_pct"),
        ],
    }
)

print(f"Complete-case circuits: {comparison_master['original_circuit'].nunique():,}")
display(
    comparison_table.style
    .format({"2Q reduction (%)": "{:.2f}", "Depth reduction (%)": "{:.2f}"})
    .set_caption(
        f"Fixed and oracle strategies on the {comparison_master['original_circuit'].nunique():,}-circuit hardware-aware complete-case subset"
    )
    .set_properties(**{"font-size": "11pt"})
)
